In [ ]:
# DAY 16 - CELL 1
# Load libraries and find the dataset

import os
import json
import random
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from google.colab import drive

drive.mount('/content/drive')

BASE_PATH = None

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for folder in dirs:
        if folder == "Underwater-Image-Data-set-main":
            BASE_PATH = os.path.join(root, folder)
            break
    if BASE_PATH:
        break

print("Dataset path:", BASE_PATH)
print("Path exists:", os.path.exists(BASE_PATH))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Mounted at /content/drive
Dataset path: /content/drive/MyDrive/Underwater-Image-Data-set-main
Path exists: True
Device: cpu


In [ ]:
# DAY 16 - CELL 2
# Create or load the frozen Dataset V1 split

pairs = []

for root, dirs, files in os.walk(BASE_PATH):
    for file in files:

        if "_input" in file.lower() and file.lower().endswith((".png", ".jpg", ".jpeg")):

            input_path = os.path.join(root, file)
            sample_id = file.rsplit("_input", 1)[0]

            target_path = os.path.join(
                root,
                sample_id + "_target.png"
            )

            if os.path.exists(target_path):
                pairs.append((input_path, target_path))

pairs = sorted(pairs)

split_path = os.path.join(
    BASE_PATH,
    "Dataset_V1_frozen_split.json"
)

if os.path.exists(split_path):

    with open(split_path, "r") as f:
        saved_split = json.load(f)

    train_pairs = [tuple(x) for x in saved_split["train"]]
    val_pairs = [tuple(x) for x in saved_split["validation"]]
    test_pairs = [tuple(x) for x in saved_split["test"]]

    print("Frozen Dataset V1 split loaded.")

else:

    random.seed(42)

    shuffled_pairs = pairs.copy()
    random.shuffle(shuffled_pairs)

    total = len(shuffled_pairs)

    train_end = int(0.70 * total)
    val_end = train_end + int(0.15 * total)

    train_pairs = shuffled_pairs[:train_end]
    val_pairs = shuffled_pairs[train_end:val_end]
    test_pairs = shuffled_pairs[val_end:]

    saved_split = {
        "train": train_pairs,
        "validation": val_pairs,
        "test": test_pairs
    }

    with open(split_path, "w") as f:
        json.dump(saved_split, f, indent=4)

    print("New Dataset V1 split created and saved.")

print("Total pairs      :", len(pairs))
print("Training pairs   :", len(train_pairs))
print("Validation pairs :", len(val_pairs))
print("Testing pairs    :", len(test_pairs))
print("Split file:", split_path)

Frozen Dataset V1 split loaded.
Total pairs      : 1233
Training pairs   : 863
Validation pairs : 184
Testing pairs    : 186
Split file: /content/drive/MyDrive/Underwater-Image-Data-set-main/Dataset_V1_frozen_split.json


In [ ]:
# DAY 16 - CELL 3
# Create PyTorch Dataset

class UnderwaterDataset(Dataset):

    def __init__(self, pairs):
        self.pairs = pairs

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, index):

        input_path, target_path = self.pairs[index]

        input_image = Image.open(input_path).convert("RGB")
        target_image = Image.open(target_path).convert("RGB")

        input_image = input_image.resize((256, 256))
        target_image = target_image.resize((256, 256))

        input_array = np.array(
            input_image,
            dtype=np.float32
        ) / 255.0

        target_array = np.array(
            target_image,
            dtype=np.float32
        ) / 255.0

        input_tensor = torch.from_numpy(
            input_array
        ).permute(2, 0, 1)

        target_tensor = torch.from_numpy(
            target_array
        ).permute(2, 0, 1)

        return input_tensor, target_tensor


train_dataset = UnderwaterDataset(train_pairs)
val_dataset = UnderwaterDataset(val_pairs)
test_dataset = UnderwaterDataset(test_pairs)

print("Training dataset:", len(train_dataset))
print("Validation dataset:", len(val_dataset))
print("Testing dataset:", len(test_dataset))

Training dataset: 863
Validation dataset: 184
Testing dataset: 186


In [ ]:
# DAY 16 - CELL 4
# Create DataLoaders

train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=8,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print("Training batches   :", len(train_loader))
print("Validation batches :", len(val_loader))
print("Testing batches    :", len(test_loader))

Training batches   : 108
Validation batches : 23
Testing batches    : 24


In [ ]:
# DAY 16 - CELL 5
# Define U-Net

class DoubleConv(nn.Module):

    def __init__(self, in_channels, out_channels):
        super().__init__()

        self.conv = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1),
            nn.ReLU(inplace=True)
        )

    def forward(self, x):
        return self.conv(x)


class UNet(nn.Module):

    def __init__(self):
        super().__init__()

        self.down1 = DoubleConv(3, 32)
        self.pool1 = nn.MaxPool2d(2)

        self.down2 = DoubleConv(32, 64)
        self.pool2 = nn.MaxPool2d(2)

        self.down3 = DoubleConv(64, 128)
        self.pool3 = nn.MaxPool2d(2)

        self.middle = DoubleConv(128, 256)

        self.up3 = nn.ConvTranspose2d(256, 128, 2, stride=2)
        self.conv3 = DoubleConv(256, 128)

        self.up2 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.conv2 = DoubleConv(128, 64)

        self.up1 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.conv1 = DoubleConv(64, 32)

        self.final = nn.Conv2d(32, 3, 1)

    def forward(self, x):

        x1 = self.down1(x)
        x2 = self.down2(self.pool1(x1))
        x3 = self.down3(self.pool2(x2))

        middle = self.middle(self.pool3(x3))

        y3 = self.up3(middle)
        y3 = torch.cat([y3, x3], dim=1)
        y3 = self.conv3(y3)

        y2 = self.up2(y3)
        y2 = torch.cat([y2, x2], dim=1)
        y2 = self.conv2(y2)

        y1 = self.up1(y2)
        y1 = torch.cat([y1, x1], dim=1)
        y1 = self.conv1(y1)

        output = self.final(y1)

        output = torch.sigmoid(output)

        return output


model = UNet().to(device)

print("U-Net created successfully.")

U-Net created successfully.


In [ ]:
# DAY 16 - CELL 6
# Test U-Net output shape

sample_input, sample_target = next(iter(train_loader))

sample_input = sample_input.to(device)

with torch.no_grad():
    sample_output = model(sample_input)

print("Input shape :", sample_input.shape)
print("Output shape:", sample_output.shape)
print("Output min  :", sample_output.min().item())
print("Output max  :", sample_output.max().item())

Input shape : torch.Size([8, 3, 256, 256])
Output shape: torch.Size([8, 3, 256, 256])
Output min  : 0.49624717235565186
Output max  : 0.5381593704223633


In [ ]:
# DAY 16 - CELL 7
# Define loss function and optimizer

criterion = nn.L1Loss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

print("Loss function: L1 Loss")
print("Optimizer: Adam")
print("Learning rate: 0.001")

Loss function: L1 Loss
Optimizer: Adam
Learning rate: 0.001


In [ ]:
# DAY 16 - CELL 8
# Train U-Net

num_epochs = 10

train_losses = []
val_losses = []

best_val_loss = float("inf")

model_path = os.path.join(
    BASE_PATH,
    "UNet_Dataset_V1_best.pth"
)

for epoch in range(num_epochs):

    model.train()

    running_train_loss = 0.0

    for inputs, targets in train_loader:

        inputs = inputs.to(device)
        targets = targets.to(device)

        optimizer.zero_grad()

        outputs = model(inputs)

        loss = criterion(outputs, targets)

        loss.backward()

        optimizer.step()

        running_train_loss += loss.item()

    average_train_loss = (
        running_train_loss / len(train_loader)
    )

    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for inputs, targets in val_loader:

            inputs = inputs.to(device)
            targets = targets.to(device)

            outputs = model(inputs)

            loss = criterion(outputs, targets)

            running_val_loss += loss.item()

    average_val_loss = (
        running_val_loss / len(val_loader)
    )

    train_losses.append(average_train_loss)
    val_losses.append(average_val_loss)

    print(
        f"Epoch {epoch + 1}/{num_epochs} "
        f"- Train Loss: {average_train_loss:.4f} "
        f"- Val Loss: {average_val_loss:.4f}"
    )

    if average_val_loss < best_val_loss:

        best_val_loss = average_val_loss

        torch.save(
            model.state_dict(),
            model_path
        )

        print("Best model saved.")

print("\nTraining completed.")
print("Best validation loss:", best_val_loss)
print("Model saved at:", model_path)

Epoch 1/10 - Train Loss: 0.1915 - Val Loss: 0.1911
Best model saved.
Epoch 2/10 - Train Loss: 0.1878 - Val Loss: 0.1936
Epoch 3/10 - Train Loss: 0.1826 - Val Loss: 0.1877
Best model saved.
Epoch 4/10 - Train Loss: 0.1818 - Val Loss: 0.1853
Best model saved.
Epoch 5/10 - Train Loss: 0.1798 - Val Loss: 0.1832
Best model saved.


In [ ]:
# DAY 16 - CELL 9
# Plot training and validation loss

plt.figure(figsize=(8, 5))

plt.plot(
    range(1, num_epochs + 1),
    train_losses,
    marker="o",
    label="Training Loss"
)

plt.plot(
    range(1, num_epochs + 1),
    val_losses,
    marker="o",
    label="Validation Loss"
)

plt.xlabel("Epoch")
plt.ylabel("L1 Loss")
plt.title("U-Net Training and Validation Loss")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# DAY 16 - CELL 10
# Generate validation outputs

model.load_state_dict(
    torch.load(model_path, map_location=device)
)

model.eval()

validation_outputs = []

with torch.no_grad():

    for inputs, targets in val_loader:

        inputs = inputs.to(device)

        outputs = model(inputs)

        validation_outputs.append(
            (
                inputs.cpu(),
                outputs.cpu(),
                targets
            )
        )

print("Validation outputs generated.")

In [ ]:
# DAY 16 - CELL 11
# Save sample validation outputs

output_folder = os.path.join(
    BASE_PATH,
    "Day16_Validation_Outputs"
)

os.makedirs(output_folder, exist_ok=True)

count = 0

for inputs, outputs, targets in validation_outputs:

    for i in range(inputs.shape[0]):

        output_image = outputs[i].permute(
            1, 2, 0
        ).numpy()

        output_image = np.clip(
            output_image * 255,
            0,
            255
        ).astype(np.uint8)

        output_path = os.path.join(
            output_folder,
            f"validation_output_{count + 1}.png"
        )

        Image.fromarray(output_image).save(
            output_path
        )

        count += 1

        if count >= 20:
            break

    if count >= 20:
        break

print("Saved validation outputs:", count)
print("Output folder:", output_folder)

In [ ]:
# DAY 16 - CELL 12
# Visualize Input vs U-Net Output vs Target

inputs, outputs, targets = validation_outputs[0]

plt.figure(figsize=(12, 12))

for i in range(min(5, inputs.shape[0])):

    input_image = inputs[i].permute(
        1, 2, 0
    ).numpy()

    output_image = outputs[i].permute(
        1, 2, 0
    ).numpy()

    target_image = targets[i].permute(
        1, 2, 0
    ).numpy()

    plt.subplot(5, 3, i * 3 + 1)
    plt.imshow(input_image)
    plt.axis("off")
    plt.title("Input")

    plt.subplot(5, 3, i * 3 + 2)
    plt.imshow(output_image)
    plt.axis("off")
    plt.title("U-Net Output")

    plt.subplot(5, 3, i * 3 + 3)
    plt.imshow(target_image)
    plt.axis("off")
    plt.title("Target")

plt.tight_layout()
plt.show()